# 뉴델리 대기질 조기경보: Aurora 1.5 + Aurora Air Pollution

이 노트북은 [`STORYBOARD_kor.md`](STORYBOARD_kor.md)의 장면을 순서대로 실행합니다.

- **S1~S2**: CAMS·ERA5 초기장으로 Aurora 0.4° Air Pollution(12시간 × 4)과 Aurora 1.5(시간별 서브스텝, 48시간)를 실행합니다. 로컬 GPU 또는 Azure AI Foundry의 "Aurora"/"Aurora-1.5" 엔드포인트를 사용할 수 있습니다.
- **S3**: 델리 NCR 10개 지점으로 보간하고 CPCB AQI, 주 오염물질, GRAP 단계, 정체 시간대(풍속 < 10 km/h, 혼합층 < 500 m)를 계산합니다.
- **S4~S5**: 힌디어/영어/한국어 시민 메시지와 운영자 브리핑을 생성해 저장합니다.

추가 패키지: `pip install cdsapi matplotlib` (demo 모드에서는 matplotlib만 선택사항).

> Aurora 1.5는 기상 모델이라 오염물질을 예측하지 않습니다. 농도는 Air Pollution 모델이, 시간별 확산 조건은 1.5가 담당합니다. 자세한 근거는 스토리보드 §2를 보세요.


In [ ]:
import sys
from pathlib import Path

# 이 노트북과 같은 폴더의 파이프라인 모듈을 불러옵니다.
# Colab처럼 저장소 없이 노트북만 열었다면 패키지와 모듈을 GitHub에서 받아옵니다.
sys.path.insert(0, str(Path.cwd()))
try:
    import delhi_aq  # noqa: F401
except ModuleNotFoundError:
    import subprocess
    import urllib.request

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "microsoft-aurora", "matplotlib"], check=True)
    RAW = "https://raw.githubusercontent.com/playground-for-thinker/aurora/main/examples/delhi_air_quality/"
    urllib.request.urlretrieve(RAW + "delhi_aq.py", "delhi_aq.py")

import pandas as pd

import delhi_aq
from delhi_aq import DELHI_SITES, plot_overview, run_pipeline, write_outputs

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)


## 1. 설정

`MODE`를 고르세요.

| MODE | 동작 |
| - | - |
| `"demo"` | 다운로드·GPU 없이 합성 데이터로 후처리만 실행 |
| `"local"` | CAMS/ERA5를 내려받아 두 모델을 로컬 GPU에서 실행 (체크포인트 수 GB 다운로드) |
| `"foundry"` | 데이터는 로컬에서 준비하고 추론은 Azure AI Foundry에서 실행 (`FOUNDRY_ENDPOINT_AP/V15`, `FOUNDRY_TOKEN_AP/V15`, `BLOB_URL_WITH_SAS`) |

데이터 다운로드에는 `ADS_API_KEY`(CAMS), `CDS_API_KEY`(ERA5) 환경 변수가 필요합니다.


In [ ]:
from datetime import datetime

MODE = "demo"  # "demo" | "local" | "foundry"
INIT_DATE = datetime(2025, 11, 5)  # 12 UTC(= 17:30 IST) 초기장
HOURS = 48
LANGS = ("ko", "en", "hi")
OUT_DIR = Path("outputs")

print(f"{MODE=}, init {INIT_DATE:%Y-%m-%d} 12 UTC, horizon {HOURS} h")
print("Sites:", ", ".join(s.name for s in DELHI_SITES))


## 2. 예측 실행 (S1~S2)

`run_pipeline`은 초기장 다운로드 → `Batch` 구성 → 모델 롤아웃 → 지점 추출 → 융합 → 메시지 생성까지 한 번에 수행합니다. 단계별로 직접 제어하려면 `delhi_aq.download_cams`, `build_air_pollution_batch`, `run_air_pollution`, `download_era5_for_v15`, `build_v15_batch`, `run_v15`, `extract_pollutants`, `extract_meteorology`, `fuse`, `summarise`를 순서대로 호출하면 됩니다.


In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

result = run_pipeline(INIT_DATE, hours=HOURS, mode=MODE, langs=LANGS)

print(len(result.pollutants), "pollutant rows (12-hourly incl. t0),", len(result.meteorology), "hourly met rows")


## 3. 지점별 요약 (S3)

정점 AQI와 그 시각(IST), 주 오염물질, GRAP 단계, 정체 시간, 추세입니다.


In [ ]:
cols = [
    "site", "kind", "aqi_now", "aqi_peak", "category_peak", "peak_time_ist", "dominant",
    "grap_stage", "stagnant_hours", "stagnant_night_hours", "min_blh_m", "mean_wind_ms", "trend",
]
summary = result.summary[cols].copy()
num = summary.select_dtypes("number").columns
summary[num] = summary[num].round(1)
summary


In [ ]:
# 시간별 융합 결과(일부): 농도는 12시간 모델의 선형 보간, blh/풍속/정체 여부는 Aurora 1.5의 시간별 값입니다.
result.fused[result.fused["site"] == "Anand Vihar"][
    ["time_ist", "pm2p5", "pm10", "aqi", "category", "blh_m", "wind10_ms", "stagnant"]
].head(24).round({"pm2p5": 0, "pm10": 0, "aqi": 0, "blh_m": 0, "wind10_ms": 1})


## 4. 시각화 (S3)

위: PM2.5(CPCB 범주 경계선 표시). 가운데: 경계층 높이(500 m 기준선). 아래: 10 m 풍속(10 km/h 기준선). 밤~새벽에 경계층이 낮고 바람이 약해지는 구간이 정체 시간대입니다.


In [ ]:
try:
    fig = plot_overview(result)
except ModuleNotFoundError:
    print("matplotlib이 없어 그래프를 건너뜁니다: pip install matplotlib")


## 5. 시민 메시지와 운영자 브리핑 (S4~S5)

`short`는 푸시/SMS용(약 160자), `long`은 앱 상세·브리핑용입니다. 템플릿은 `delhi_aq.HEALTH_ADVICE`, `GRAP_ADVICE`, `CATEGORY_NAMES`에 있어 보건 전문가가 코드 변경 없이 검수할 수 있습니다.


In [ ]:
for lang in LANGS:
    print(f"[{lang}] {result.city[lang]}")
print()

worst = result.summary.loc[result.summary["aqi_peak"].idxmax(), "site"]
for lang in LANGS:
    m = next(x for x in result.messages[lang] if x["site"] == worst)
    print(f"--- {lang} / short ({len(m['short'])} chars) ---")
    print(m["short"])
    print()
print(f"--- ko / long ({worst}) ---")
print(next(x for x in result.messages["ko"] if x["site"] == worst)["long"])


## 6. 저장과 발송 연계

`write_outputs`는 CSV, 언어별 `messages_*.json`, `bulletin_*.md`를 저장합니다. 발송은 이 JSON을 읽어 채널(WhatsApp Business API, SMS 게이트웨이, 앱 푸시)로 넘기는 작은 어댑터만 추가하면 됩니다. 사용자의 거주 지역(핀코드)을 가장 가까운 `Site`에 매핑하는 것이 일반적입니다.


In [ ]:
paths = write_outputs(result, OUT_DIR)
for p in paths:
    print(p)


### 지도에서 보기 (msresearch-aurora 프런트엔드)

`outputs/frontend/`에는 델리 NCR 타일(27.5~31.5°N, 76~80°E, 0.25°)의 시간별 격자가 `msresearch-aurora` 노르웨이 예제의 프런트엔드 스키마로 저장됩니다. 지도 실행 절차는 [`README_kor.md`](README_kor.md)의 "msresearch-aurora 스킬과의 연계"를 보세요. 요약:

```bash
cd msresearch-aurora/assets/inference/scripts
python3 setup_region.py --name "Delhi NCR" --lat-min 27.5 --lat-max 31.5 --lon-min 76 --lon-max 80 --skip-download
cp examples/delhi_air_quality/outputs/frontend/*.json msresearch-aurora/assets/inference/delhi-ncr-example/frontend/public/data/
cd msresearch-aurora/assets/inference/delhi-ncr-example/frontend && npm install && npm run dev
```


In [ ]:
# 타일 요약: 시간별 최대 AQI와 PM2.5 평균 (격자 전체)
import numpy as np

g = result.grid
tile = pd.DataFrame(
    {
        "time_utc": g.times,
        "pm2p5_mean": g.fields["pm2p5"].mean(axis=(1, 2)).round(0),
        "aqi_max": np.nanmax(g.fields["aqi"], axis=(1, 2)).round(0),
        "wind_mean_ms": g.fields["wind_ms"].mean(axis=(1, 2)).round(1),
    }
)
tile.iloc[::6]


## 7. 한계와 다음 단계

- Air Pollution 모델은 0.4° 해상도이며 CAMS 분석 전용입니다. 지점별 수치는 보간 결과이고 도로변 농도가 아닙니다.
- 시간별 농도 곡선은 12시간 예측의 선형 보간이며, 시간별 구조의 근거는 Aurora 1.5의 기상 변수입니다.
- Aurora 1.5는 IFS HRES T0로 파인튜닝되었고 이 예제는 ERA5를 쓰므로 정확도가 보장되지 않습니다.
- 운영 사용 전에는 CPCB 관측과의 검증·편향 보정, 보건 전문가의 문구 검수, 그리고 CAQM 공식 단계와의 정합성 확인이 필요합니다.

다음 단계로는 `AuroraV1p5Ensemble` + `rollout_ensemble`로 정체 확률을 표현하거나, 인도 지역 자료로 Air Pollution 모델을 파인튜닝하는 것을 스토리보드 §7에 정리했습니다.
